# Dia 2 — Memória Persistente + Múltiplas Tools

Este notebook é **autossuficiente** — você não precisa do notebook do Dia 1 aberto.

| Parte | Tema |
|---|---|
| Setup | Dependências, credenciais, LLM e e-mail |
| Memória | `SqliteSaver` + `invocar_com_memoria()` |
| Tools | 5 tools de e-mail — definição e teste direto |
| Agente | Um único agente com todas as tools |
| Sem memória | `invocar_sem_memoria()` — cada chamada começa do zero |
| Thread isolation | Mesmo banco, históricos separados por `thread_id` |

---

## Setup

In [ ]:
!pip install -q langchain-anthropic langchain langgraph aiosqlite

In [57]:
# Preencha com os valores que o professor forneceu
PROXY_URL      = "https://interview-server-mocado.b60gda.easypanel.host/"
ALUNO_TOKEN    = "xpto_aluno-01"   # token do proxy LLM

EMAIL_URL      = "https://interview-email-server.b60gda.easypanel.host/"
EMAIL_TOKEN    = "aluno-01"
EMAIL_PASSWORD = "1234"

print("Credenciais carregadas.")

Credenciais carregadas.


In [58]:
import requests

r = requests.get(f"{PROXY_URL}/health")
print(r.status_code, r.json())

200 {'status': 'ok', 'anthropic_key_configured': True, 'alunos_registrados': 10}


In [59]:
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage

llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    api_key=ALUNO_TOKEN,
    base_url=PROXY_URL,
    max_tokens=512,
)

resposta = llm.invoke([HumanMessage(content="Responda apenas: conexão ok!")])
print(resposta.content)

conexão ok!


In [60]:
r = requests.post(f"{EMAIL_URL}/auth/login", json={
    "token": EMAIL_TOKEN,
    "password": EMAIL_PASSWORD,
})
assert r.status_code == 200, f"Login falhou: {r.json()}"
_headers = {"Authorization": f"Bearer {EMAIL_TOKEN}"}
print("Login OK →", r.json())

Login OK → {'token': 'aluno-01', 'name': 'Aluno 1', 'email': 'aluno01@curso.ia'}


In [61]:
# Verificar caixa de entrada diretamente (sem agente)
r = requests.get(f"{EMAIL_URL}/emails/inbox", headers=_headers)
inbox = r.json()
print(f"{inbox['count']} mensagem(ns) para {inbox['email']}")
for msg in inbox["messages"]:
    print(f"  [{msg['id']}] De: {msg['from']} | {msg['subject']}")

0 mensagem(ns) para aluno01@curso.ia


---
## Memória Persistente — SqliteSaver

Configuramos a memória **antes** de criar qualquer agente. Assim todos os agentes nascem com ela.

> 💡 `SqliteSaver` grava o histórico em um arquivo `.db`. No Google Colab, salvamos no **Google Drive** — a memória sobrevive mesmo que o ambiente reinicie.  
> Cada `thread_id` é uma sessão isolada — usaremos o token do aluno como `thread_id`.

In [62]:
import os
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/curso_agentes", exist_ok=True)
    DB_PATH = "/content/drive/MyDrive/curso_agentes/memoria.db"
    print(f"Google Drive montado. Banco: {DB_PATH}")
except ImportError:
    DB_PATH = "memoria.db"
    print(f"Executando localmente. Banco: {DB_PATH}")

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
checkpointer = SqliteSaver(conn)

def invocar_com_memoria(prompt: str, thread_id: str = None) -> str:
    """Invoca o agente com memória persistente.

    thread_id identifica a sessão — por padrão usa o token do aluno.
    Mesma thread retoma de onde parou; thread diferente = histórico zerado.
    """
    tid = thread_id or EMAIL_TOKEN
    resultado = agente.invoke(
        {"messages": [{"role": "user", "content": prompt}]},
        config={"configurable": {"thread_id": tid}, "recursion_limit": 10},
    )
    return resultado["messages"][-1].content

print("checkpointer e invocar_com_memoria() prontos.")

Executando localmente. Banco: memoria.db
checkpointer e invocar_com_memoria() prontos.


---
## Tools de e-mail

Cinco tools que o agente poderá usar. A **docstring** de cada uma é o que o modelo lê para decidir quando e como usá-la.

Testamos cada tool **diretamente** (sem agente) para confirmar que funciona antes de conectar.

In [63]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from typing import Optional

@tool
def send_email(to: str, subject: str, body: str, cc: Optional[str] = None) -> str:
    """Envia um e-mail pelo servidor do curso.

    Use esta tool quando o usuário pedir para enviar ou responder um e-mail.

    Args:
        to: E-mail do destinatário (ex: aluno02@curso.ia)
        subject: Assunto do e-mail
        body: Corpo completo do e-mail
        cc: E-mails em cópia, separados por vírgula (opcional)
    """
    payload = {"to": to, "subject": subject, "body": body}
    if cc:
        payload["cc"] = cc
    r = requests.post(f"{EMAIL_URL}/emails/send", headers=_headers, json=payload)
    if r.status_code in (200, 201):
        return f"E-mail enviado com sucesso para '{to}'."
    return f"Erro ao enviar ({r.status_code}): {r.json()}"

print("Tool send_email definida.")

Tool send_email definida.


In [64]:
@tool
def get_inbox() -> str:
    """Lista os e-mails na caixa de entrada do usuário.

    Retorna id, data, remetente e assunto de cada mensagem.
    Use esta tool quando o usuário quiser saber o que tem na caixa de entrada.
    """
    r = requests.get(f"{EMAIL_URL}/emails/inbox", headers=_headers)
    inbox = r.json()
    if inbox["count"] == 0:
        return "A caixa de entrada está vazia."
    linhas = [f"Total: {inbox['count']} mensagem(ns)\n"]
    for msg in inbox["messages"]:
        linhas.append(
            f"ID: {msg['id']} | Data: {msg['timestamp']} | "
            f"De: {msg['from']} | Assunto: {msg['subject']}"
        )
    return "\n".join(linhas)

print(get_inbox.invoke({}))

A caixa de entrada está vazia.


In [65]:
@tool
def get_sent() -> str:
    """Lista os e-mails enviados pelo usuário.

    Retorna id, data, destinatário e assunto de cada mensagem enviada.
    Use esta tool quando o usuário quiser ver os e-mails que já enviou.
    """
    r = requests.get(f"{EMAIL_URL}/emails/sent", headers=_headers)
    sent = r.json()
    if sent["count"] == 0:
        return "Nenhum e-mail enviado ainda."
    linhas = [f"Total: {sent['count']} mensagem(ns) enviada(s)\n"]
    for msg in sent["messages"]:
        destinatarios = ", ".join(msg["to"]) if isinstance(msg["to"], list) else msg["to"]
        linhas.append(
            f"ID: {msg['id']} | Data: {msg['timestamp']} | "
            f"Para: {destinatarios} | Assunto: {msg['subject']}"
        )
    return "\n".join(linhas)

print(get_sent.invoke({}))

Total: 4 mensagem(ns) enviada(s)

ID: ddc3a4a5-1e25-4ed0-a5d4-2bf79ae7da28 | Data: 2026-04-22T18:41:08.450531+00:00 | Para: aluno10@curso.ia | Assunto: Confirmação de Reunião - Amanhã
ID: 1e9145f4-1405-4db9-b0f4-98215b7553b7 | Data: 2026-04-22T18:36:50.491003+00:00 | Para: aluno10@curso.ia | Assunto: Confirmação de Reunião - Amanhã às 10:00h
ID: 458af541-007c-4f68-8d85-7f1159f90c30 | Data: 2026-04-22T18:15:35.682241+00:00 | Para: aluno02@curso.ia | Assunto: Teste Dia 2
ID: ef1a4434-a325-4edc-a2d1-8044987a8a9f | Data: 2026-04-22T17:09:34.778877+00:00 | Para: aluno02@curso.ia | Assunto: Teste manual


In [66]:
@tool
def get_email(email_id: str) -> str:
    """Lê o conteúdo completo de um e-mail específico.

    Use esta tool quando o usuário quiser ler ou responder um e-mail.
    Você precisa do ID — obtenha-o com get_inbox primeiro.

    Args:
        email_id: ID do e-mail a ser lido.
    """
    r = requests.get(f"{EMAIL_URL}/emails/{email_id}", headers=_headers)
    if r.status_code == 404:
        return f"E-mail com ID '{email_id}' não encontrado."
    msg = r.json()
    return (
        f"De:      {msg['from']}\n"
        f"Para:    {msg.get('to', '')}\n"
        f"CC:      {msg.get('cc', '')}\n"
        f"Assunto: {msg['subject']}\n"
        f"Data:    {msg['timestamp']}\n"
        f"\n{msg['body']}"
    )

# Teste: substitua pelo ID de um e-mail real
# print(get_email.invoke({"email_id": "COLE_UM_ID_AQUI"}))
print("Tool get_email definida.")

Tool get_email definida.


In [67]:
@tool
def search_emails(query: str) -> str:
    """Busca e-mails na caixa de entrada por remetente ou palavras no assunto.

    Use esta tool quando o usuário perguntar sobre e-mails de uma pessoa
    específica ou com um assunto específico.

    Args:
        query: Termo de busca — nome, e-mail do remetente ou palavra no assunto.
    """
    r = requests.get(f"{EMAIL_URL}/emails/inbox", headers=_headers)
    inbox = r.json()
    q = query.lower()
    encontrados = [
        msg for msg in inbox["messages"]
        if q in msg["from"].lower() or q in msg["subject"].lower()
    ]
    if not encontrados:
        return f"Nenhum e-mail encontrado para a busca: '{query}'."
    linhas = [f"{len(encontrados)} e-mail(s) encontrado(s) para '{query}':\n"]
    for msg in encontrados:
        linhas.append(f"ID: {msg['id']} | De: {msg['from']} | Assunto: {msg['subject']}")
    return "\n".join(linhas)

print(search_emails.invoke({"query": "reunião"}))

Nenhum e-mail encontrado para a busca: 'reunião'.


---
## Agente

Um único agente com todas as 5 tools e memória persistente via `checkpointer`.

In [68]:
todas_as_tools = [get_inbox, get_sent, get_email, search_emails, send_email]

agente = create_agent(
    model=llm,
    tools=todas_as_tools,
    system_prompt=(
        "Você é um assistente de e-mail. "
        "Use as ferramentas disponíveis para ler, buscar, listar enviados e enviar e-mails. "
        "Os endereços seguem o padrão alunoXX@curso.ia (ex: aluno02@curso.ia). "
        "Seja objetivo nas respostas."
    ),
    checkpointer=checkpointer,
)

print(f"Agente criado com {len(todas_as_tools)} tools:", [t.name for t in todas_as_tools])

Agente criado com 5 tools: ['get_inbox', 'get_sent', 'get_email', 'search_emails', 'send_email']


## Testes com o Agente com memoria  -  `invocar_com_memoria()`

In [69]:
# Teste 1 — listar inbox
print(invocar_com_memoria("O que tem na minha caixa de entrada?"))

Sua caixa de entrada está vazia! 📭 Você não tem nenhum e-mail no momento.


In [70]:
# Teste 2 — busca por assunto
print(invocar_com_memoria("Tem algum e-mail sobre reunião?"))

Não encontrei nenhum e-mail sobre "reunião" na sua caixa de entrada. 📧


In [71]:
# Teste 3 — sequência de tools (search → get_email)
print(invocar_com_memoria("Encontre e-mails do aluno05 e me diga o que o último diz."))

Não encontrei nenhum e-mail do aluno05 na sua caixa de entrada. 📧


In [77]:
# Teste 1 — listar inbox
print(invocar_com_memoria("Envie um email par ao aluno05 confirmando nossa reunião para aamnhã as 10:00."))

Pronto! ✅ E-mail enviado com sucesso para aluno05@curso.ia confirmando a reunião para amanhã às 10:00.


In [78]:
# Teste 2 — busca por assunto
print(invocar_com_memoria("Acabei de lembrar que tenho compromisso, vamos desmarcar a reunião."))

Feito! ✅ E-mail enviado para aluno05@curso.ia desmarcando a reunião de amanhã às 10:00.


---
## Sem memória — `invocar_sem_memoria()`

Usa um `thread_id` aleatório a cada chamada — o agente nunca encontra histórico anterior.  
Útil para entender a diferença em relação a `invocar_com_memoria()`.

In [72]:
import uuid

def invocar_sem_memoria(prompt: str, ag=agente) -> str:
    """Invoca o agente sem memória — cada chamada é independente."""
    resultado = ag.invoke(
        {"messages": [{"role": "user", "content": prompt}]},
        config={"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 10},
    )
    return resultado["messages"][-1].content

print("invocar_sem_memoria() pronto.")

invocar_sem_memoria() pronto.


In [75]:
# Teste 1 — listar inbox
print(invocar_sem_memoria("Envie um email par ao aluno05 confirmando nossa reunião para aamnhã as 10:00."))

E-mail enviado com sucesso para aluno05@curso.ia confirmando a reunião de amanhã às 10:00!


In [76]:
# Teste 2 — busca por assunto
print(invocar_sem_memoria("Acabei de lembrar que tenho compromisso, vamos desmarcar a reunião."))

Entendo que você quer desmarcar uma reunião, mas para ajudá-lo melhor, preciso de mais informações:

1. **Qual é o e-mail do organizador ou participante principal da reunião?** (para eu encontrar a mensagem)
2. **Qual é o assunto ou tema da reunião?** (para facilitar a busca)

Assim poderei buscar o e-mail sobre essa reunião e você poderá enviar uma mensagem desmarcando-a.


In [ ]:
# Verbose — ver o chain-of-thought interno
# Mostra TODAS as mensagens trocadas, incluindo chamadas de tool e respostas.

def invocar_verbose(prompt, ag=agente):
    resultado = ag.invoke(
        {"messages": [{"role": "user", "content": prompt}]},
        config={"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 10},
    )
    for msg in resultado["messages"]:
        tipo = msg.__class__.__name__
        print(f"\n{'='*40}")
        print(f"  {tipo}")
        print(f"{'='*40}")
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  → tool: {tc['name']}")
                print(f"     args: {tc['args']}")
        elif hasattr(msg, "content"):
            conteudo = str(msg.content)
            print(conteudo[:400] + ("..." if len(conteudo) > 400 else ""))

invocar_verbose("Tem algum e-mail sobre reunião? Se sim, me diga o remetente.")

---
## Thread Isolation

O `SqliteSaver` grava o histórico em um arquivo `.db`. Cada `thread_id` é uma sessão independente — usuários diferentes têm históricos completamente separados, mesmo usando o mesmo agente e o mesmo banco.

Simulamos dois alunos com o **mesmo agente**, **mesmo banco**, mas `thread_id` diferentes.

In [84]:
print("=== Aluno A — Turno 1 ===")
print(invocar_com_memoria("O que tem na minha caixa de saida?", thread_id="aluno-01"))

=== Aluno A — Turno 1 ===
Sua caixa de saída agora tem **8 e-mails enviados**:

1. **Convite para Cafezinho** - Para: aluno05@curso.ia (22/04/2026 às 19:14:17) ⭐ *Novo*
2. **Desmarcação de Reunião** - Para: aluno05@curso.ia (22/04/2026 às 19:10:32)
3. **Confirmação de Reunião** - Para: aluno05@curso.ia (22/04/2026 às 19:10:25)
4. **Confirmação de Reunião - Amanhã às 10:00** - Para: aluno05@curso.ia (22/04/2026 às 19:08:50)
5. **Confirmação de Reunião - Amanhã** - Para: aluno10@curso.ia (22/04/2026 às 18:41:08)
6. **Confirmação de Reunião - Amanhã às 10:00h** - Para: aluno10@curso.ia (22/04/2026 às 18:36:50)
7. **Teste Dia 2** - Para: aluno02@curso.ia (22/04/2026 às 18:15:35)
8. **Teste manual** - Para: aluno02@curso.ia (22/04/2026 às 17:09:34)


In [81]:
print("=== Aluno A — Turno 2 ===")
print(invocar_com_memoria("Leia o primeiro e-mail da lista.", thread_id="aluno-01"))

=== Aluno A — Turno 2 ===
Aqui está o conteúdo do primeiro e-mail:

**De:** aluno01@curso.ia  
**Para:** aluno05@curso.ia  
**Assunto:** Desmarcação de Reunião  
**Data:** 22/04/2026 às 19:10:32

**Corpo:**
```
Olá!

Infelizmente, surgiu um compromisso e preciso desmarcar nossa reunião que estava agendada para amanhã às 10:00.

Peço desculpas pela inconveniência. Vamos reagendar em breve.

Obrigado pela compreensão!
```

Este é o e-mail que acabei de enviar desmarcando a reunião! 📧


In [82]:
print("=== Aluno A — Turno 3 ===")
print(invocar_com_memoria("Mande um email para este mesmo destinatário para marcar um cafezinho amanhã as 16:00 h.", thread_id="aluno-02"))



=== Aluno A — Turno 3 ===
Não consigo enviar o email porque você não especificou qual é o destinatário. 

Você pode me fornecer:
- O nome ou número do aluno (ex: aluno02)
- Ou o email completo do destinatário

Após isso, farei o envio do email marcando o cafezinho para amanhã às 16:00h.


In [83]:
print("=== Aluno A — Turno 3 ===")
print(invocar_com_memoria("Mande um email para este mesmo destinatário para marcar um cafezinho amanhã as 16:00 h.", thread_id="aluno-01"))

=== Aluno A — Turno 3 ===
Pronto! ✅ E-mail enviado com sucesso para aluno05@curso.ia convidando para um cafezinho amanhã às 16:00h!


---
## Resumo do Dia 2

| Conceito | O que aprendemos |
|---|---|
| `SqliteSaver` + Google Drive | Memória que persiste entre sessões — mesmo se o Colab reiniciar |
| `thread_id` | Cada usuário tem sua sessão isolada no mesmo arquivo `.db` |
| `invocar_com_memoria()` | Invoca com histórico persistente por `thread_id` |
| `invocar_sem_memoria()` | Cada chamada começa do zero — `thread_id` aleatório |
| `@tool` + docstring | A docstring é o que o modelo lê para decidir quando usar a tool |
| Teste direto de tool | `get_inbox.invoke({})` — testar sem agente facilita depuração |
| Agente multi-tool | O modelo escolhe a tool certa dado o contexto do pedido |
| Sequência de tools | O modelo pode encadear tools (`search → get_email → send_email`) |
| Verbose | Torna o chain-of-thought interno visível |
| Thread isolation | Mesmo banco, históricos completamente separados por `thread_id` |

### O que vem no Dia 3

- Mini-projeto integrado: um único prompt que lê, decide e age
- Introdução ao calendário como nova fonte de tools
- Classificação e priorização de e-mails com few-shot prompting